In [ ]:
import os
import glob
import pandas as pd
import re
import matplotlib.pyplot as plt
import spacy
import gender_guesser.detector as gender

In [ ]:
base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
csv_file = glob.glob(os.path.join(base_dir, "/data/political_guardian_articles.csv"))
df = pd.read_csv(csv_file[0])

In [ ]:
male_terms = [
    "man", "men", "male", "boy", "he", "him", "his"
]

female_terms = [
    "woman", "women", "female", "girl", "she", "her", "hers"
]

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

from collections import Counter

def count_gender_terms(article):
    tokens = tokenize(article)
    counts = Counter(tokens)
    
    male_count = sum(counts[word] for word in male_terms)
    female_count = sum(counts[word] for word in female_terms)
    
    return male_count, female_count


In [ ]:
male_total = 0
female_total = 0
for i, article in enumerate(df['bodyContent']):
    male_count, female_count = count_gender_terms(article)
    male_total += male_count
    female_total += female_count

print(f"Total male mentions in Politics: {male_total}")
print(f"Total female mentions in Politics: {female_total}")

In [ ]:
labels = ['Male Mentions', 'Female Mentions']
sizes = [male_total, female_total]
colors = ['lightblue', 'lightpink']
plt.figure(figsize=(8, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
plt.title('Gendered Word Frequency in Guardian Political Articles')
plt.axis('equal')
plt.show()

In [ ]:
detector = gender.Detector()
def absolute_gender(name):
    result = detector.get_gender(name)
    if result in ("male", "mostly_male"):
        return "male"
    elif result in ("female", "mostly_female"):
        return "female"
    else:
        return "unknown"


In [ ]:
nlp = spacy.load("en_core_web_sm")
pd.set_option("display.max_rows", 200)

results = {'male': 0, 'female': 0, 'unknown': 0}

for i in range(min(100, 400)):
    content = str(df.head(i+1)['bodyContent'].values[-1])
    doc = nlp(content)
    prev_names = set()
    
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            name = ent.text.strip()
            first_name = name.split()[0]
            second_name = name.split()[1] if len(name.split()) > 1 else ""
            result = absolute_gender(first_name)
            if name in prev_names or second_name in prev_names:
                continue
            prev_names.add(name)
            prev_names.add(second_name)
            results[result] += 1
            
            if result == "unknown":
                print(name, result)
total = sum(results.values())
for key in results:
    results[key] = results[key] / total * 100

plt.figure(figsize=(12, 6))
plt.bar(results.keys(), results.values())
plt.title('Gender Distribution of Named Entities in Guardian Political Articles')
plt.ylabel('Percentage (%)')
plt.show()